In [ ]:
# Cell 1: 学習曲線データの読み込み
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

RUN_ROOT = Path("/home/nakano/server/checkpoints_dense/vision_plain/cifar10")

def natural_ckpt_key(p: Path):
    m = re.search(r"checkpoint-(\d+)", p.name)
    return int(m.group(1)) if m else -1

def load_log_history(net_name: str):
    run_dir = RUN_ROOT / net_name
    ckpt_dirs = sorted(
        [d for d in run_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
        key=natural_ckpt_key,
    )
    latest = ckpt_dirs[-1]
    state = json.load(open(latest / "trainer_state.json"))
    entries = [e for e in state["log_history"] if "loss" in e]
    steps = np.array([e["step"] for e in entries])
    epochs = np.array([e["epoch"] for e in entries])
    losses = np.array([e["loss"] for e in entries])
    grad_norms = np.array([e.get("grad_norm", np.nan) for e in entries])
    lrs = np.array([e.get("learning_rate", np.nan) for e in entries])
    return {"steps": steps, "epochs": epochs, "loss": losses, "grad_norm": grad_norms, "lr": lrs}

logs = {net: load_log_history(net) for net in ["convnext-atto", "swin-atto"]}
for net, d in logs.items():
    print(f"{net}: {len(d['steps'])} log points, final loss={d['loss'][-1]:.4f}")

ModuleNotFoundError: No module named 'torchvision'